<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-lockup-tight.svg" width="440">

# rumi in ten minutes

**rumi** is a GeoTIFF profile for AI training data. The pixels stay in a real
BigTIFF that any reader can walk. Alongside it rumi keeps a small binary header
that says where every tile lives, so a reader jumps straight to the tiles it
wants without parsing the IFD.

Tile payloads are compressed with [OpenZL](https://github.com/facebook/openzl)
through [geozl](https://github.com/asterisk-labs/geozl), which picks a codec
graph that fits the data instead of one general purpose compressor.

This notebook builds a scene, tiles it, compresses it, writes it, and reads it
back. Run the cells top to bottom.

- Repo <https://github.com/asterisk-labs/rumi>
- Spec <https://github.com/asterisk-labs/rumi/blob/main/SPEC.md>

## Setup

geozl ships wheels for Linux x86_64 and macOS arm64. Colab is Linux x86_64, so
this works out of the box. rasterio is only here to make and read the source
image, rumi never needs it.

In [1]:
!pip install rumi-eo geozl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 38.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import rasterio
import geozl
import rumi

rumi.__version__, geozl.__version__

('0.13.0', '0.11.0')

## A scene to work with

Three bands at 5490 px, the size of a Sentinel-2 20 m tile, in UTM 18S over
Peru. Swap this cell for your own file if you have one.

We also write the same pixels as a tiled DEFLATE GeoTIFF, which is the honest
baseline to compare against later. Comparing to an uncompressed file would
flatter rumi for no reason.

In [5]:
from rasterio.transform import from_origin

N = 5490
rng = np.random.default_rng(0)
y, x = np.mgrid[0:N, 0:N].astype(np.float32)

bands = np.empty((3, N, N), np.uint16)
for i, f in enumerate((
        lambda: 2000 + 800 * np.sin(x / 90) * np.cos(y / 120),
        lambda: 1500 + 600 * np.cos(x / 70),
        lambda: 3000 + 900 * np.sin((x + y) / 110))):
    band = f() + rng.normal(0, 40, (N, N)).astype(np.float32)
    bands[i] = band.clip(0, 10000).astype(np.uint16)
del y, x, band

common = dict(driver="GTiff", height=N, width=N, count=3, dtype="uint16",
              crs="EPSG:32718", transform=from_origin(500000, 8000000, 10, 10))

with rasterio.open("demo.tif", "w", **common) as dst:
    dst.write(bands)

with rasterio.open("demo_deflate.tif", "w", **common, tiled=True,
                   blockxsize=512, blockysize=512,
                   compress="deflate", predictor=2) as dst:
    dst.write(bands)

src = rasterio.open("demo.tif")
src.profile

{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': None, 'width': 5490, 'height': 5490, 'count': 3, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 18S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-75],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32718"]]'), 'transform': Affine(10.0, 0.0, 500000.0,
       0.0, -10.0, 8000000.0), 'blockxsize': 5490, 'blockysize': 1, 'tiled': False, 'interleave': 'pixel'}

## The data model

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-data-model.svg" width="620">

A **TileFrame** is the grid. One row per tile, carrying the pixels and, once you
compress it, the frame. Edge tiles are **cut, never padded**, so a tile at the
right edge is genuinely narrower and nothing is invented or decoded that was
never written.

In [6]:
tf = rumi.tile(src.read(), 512)
tf

<rumi.TileFrame (3, 5490, 5490)>
        tile   cell     data  compressed
  0    0.0.0    0.0  512×512           ·
  1    1.0.0    0.0  512×512           ·
  2    2.0.0    0.0  512×512           ·
  3    0.0.1    0.1  512×512           ·
  4    1.0.1    0.1  512×512           ·
  …        …      …        …           …
358   1.10.9   10.9  512×370           ·
359   2.10.9   10.9  512×370           ·
360  0.10.10  10.10  370×370           ·
361  1.10.10  10.10  370×370           ·
362  2.10.10  10.10  370×370           ·

  dtype : uint16
  tile  : 512 × 512
  grid  : 11 × 11 × 3
  tiles : 363

`tile` and `cell` name the position, band first for `tile`. The band, row
and column are still there as dims when you need to route tiles somewhere.

In [7]:
tf.columns, tf.dims

(('tile', 'cell', 'data', 'compressed'), ('band', 'row', 'col'))

In [8]:
t = tf[0]
t.tile, t.cell, t.data.shape, t.compressed

('0.0.0', '0.0', (512, 512), None)

5490 is not a multiple of 512, so the last row and column come up short.
Four distinct shapes for any image, and that matters in a moment.

In [9]:
sorted({t.data.shape for t in tf})

[(370, 370), (370, 512), (512, 370), (512, 512)]

## Compressing

rumi does not compress. You pick the graph, which is the point, since a codec
that suits a DEM is not the one that suits an S1 backscatter raster.
`geozl.profile` measures the candidates on a real tile.

In [10]:
import pandas as pd
pd.DataFrame(geozl.profile(tf[0].data)).head(6)

,graph,bytes,ratio,encode_mbps,decode_mbps,shannon_pct
0,planar>zigzag>transpose>entropy,277041,1.892456,234.206973,376.498975,148.434559
1,planar>zigzag>entropy,277542,1.889040,356.924133,577.419866,148.166615
2,planar>zigzag>categorical,277542,1.889040,85.758553,560.791608,148.166615
3,planar>zigzag>store_lo,279092,1.878549,330.818016,609.850657,147.343739
4,planar>zigzag>transpose>zstd,291230,1.800254,84.471066,981.525996,141.202688
5,id>transpose>zstd,293331,1.787360,98.162027,1292.795392,140.191316


One catch worth knowing before it bites you. A graph carries the stride it
was built with, and edge tiles are shorter, so a graph built on a 512x512 tile
refuses the 370x370 corner. Keep one graph per tile shape, at most four.

In [11]:
best = geozl.profile(tf[0].data)[0]["graph"]

for t in tf:
    graph = geozl.graph(t.data, best)
    t.compressed = geozl.compress(t.data, graph=graph)
tf

<rumi.TileFrame (3, 5490, 5490)>
        tile   cell     data  compressed
  0    0.0.0    0.0  512×512    270.5 KB
  1    1.0.0    0.0  512×512    270.4 KB
  2    2.0.0    0.0  512×512    270.5 KB
  3    0.0.1    0.1  512×512    270.5 KB
  4    1.0.1    0.1  512×512    270.5 KB
  …        …      …        …           …
358   1.10.9   10.9  512×370    195.4 KB
359   2.10.9   10.9  512×370    195.6 KB
360  0.10.10  10.10  370×370    141.4 KB
361  1.10.10  10.10  370×370    141.4 KB
362  2.10.10  10.10  370×370    141.3 KB

  dtype : uint16
  tile  : 512 × 512
  grid  : 11 × 11 × 3
  tiles : 363
  bytes : 91.1 MB (1.9×)

## Writing

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-index.svg" width="700">

`rumi.write` returns the path and the **header**, a compact sidecar of raw
bytes. It is not a competing format, the file it describes is a normal BigTIFF.
The header just means a reader never has to open the file to know what is in it,
or walk the IFD to find a tile.

In [12]:
path, header = rumi.write("demo.rumi.tif", tf,
                          transform=src.transform,
                          crs=src.crs.to_epsg())

import os
raw = os.path.getsize("demo.tif")
for label, name in (("uncompressed", "demo.tif"),
                    ("GeoTIFF DEFLATE", "demo_deflate.tif"),
                    ("rumi", path)):
    n = os.path.getsize(name)
    print(f"{label:18} {n:>12,} bytes   {raw / n:4.2f}x")

print(f"\n{'header':18} {len(header):>12,} bytes")

uncompressed        180,873,924 bytes   1.00x
GeoTIFF DEFLATE     119,811,429 bytes   1.51x
rumi                 95,547,773 bytes   1.89x

header                    1,478 bytes


`transform` takes rasterio's `Affine` order, `(x_res, row_rot, x_origin,
col_rot, y_res, y_origin)`. Passing `src.transform.to_gdal()` instead writes a
georeferencing that is silently wrong, so hand it the `Affine` itself.

## The header on its own

Bytes in, answers out. No file is opened here.

In [13]:
h = rumi.RumiHeader(header)
h

dtype,uint16
tile,512 × 512
tiles,363
tiles/band,121
codec,OpenZL


In [14]:
h.to_dict()

{'shape': [3, 5490, 5490],
 'bands': 3,
 'height': 5490,
 'width': 5490,
 'dtype': 'uint16',
 'tile': [512, 512],
 'tiles_across': 11,
 'tiles_down': 11,
 'tiles': 363,
 'base_tiles_offset': 4812,
 'codec': 'OpenZL'}

## Reading

`read` takes the header bytes you cached. Leave it out and rumi reads it off the
file, which costs one extra open.

In [15]:
full = rumi.read(path, header, num_threads=4)
full.shape, full.dtype, np.array_equal(full, src.read())

((3, 5490, 5490), dtype('uint16'), True)

A window and a band subset only decode the tiles they touch. Compare the
wall time against the full read above.

In [16]:
%%time
patch = rumi.read(path, header, b=[0, 2], y=(2000, 2512), x=(3000, 3512))
patch.shape

CPU times: user 14.1 ms, sys: 4.03 ms, total: 18.2 ms
Wall time: 19.4 ms


(2, 512, 512)

`pattern` names the axis order you want out, so the transpose happens
during assembly rather than as a copy afterwards.

In [17]:
rumi.read(path, header, pattern="y x b", y=(0, 256), x=(0, 256)).shape

(256, 256, 3)

A list of paths reads a stack, with `n` as the image axis.

In [18]:
cube = rumi.read([path, path, path], [header] * 3,
                 pattern="n b y x", y=(0, 512), x=(0, 512))
cube.shape

(3, 3, 512, 512)

`framework` picks what comes back. `"numpy"` is the default, `"torch"`,
`"jax"` and `"tensorflow"` work the same way, and `None` hands you a zero copy
DLPack array that any of them can adopt.

In [19]:
arr = rumi.read(path, header, framework=None)
print(arr)

np.from_dlpack(rumi.read(path, header, framework=None)).shape

<rumi.RumiArray (3, 5490, 5490) uint16>


(3, 5490, 5490)

## Why the header is bytes

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-catalog.svg" width="700">

Because bytes go in a column. Put the header in Parquet next to the path and a
loader knows the shape, dtype and tile grid of every scene in the dataset
without touching object storage.

## Where to go next

- The spec, if you want to write a reader
  <https://github.com/asterisk-labs/rumi/blob/main/SPEC.md>
- geozl, the codec side
  <https://github.com/asterisk-labs/geozl>
- Issues and questions
  <https://github.com/asterisk-labs/rumi/issues>

<br>

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/asterisk_banner.svg" width="200">